# Notebook 04 — EDA: Model Comparison
**Question:** Which model is best to trade live?

**Decision Rule (stated before looking at results):**  
The winning model must satisfy ALL of the following on the holdout period:
1. Expectancy > 0 (bootstrap 95% CI lower bound > 0)
2. Profit Factor > 1.1
3. Max Drawdown < −15 R
4. Average trades/month ≥ 20
5. If two models tie on 1–4, prefer higher Calmar ratio.

**Hypothesis (pre-registered):**  
The rule-based model offers the best risk-adjusted return because it systematically removes low-quality setups. The ML variant may show comparable or slightly lower absolute returns but worse stability due to small sample size.

**Threats to validity:** single instrument, same backtest engine for all models, spread/slippage assumptions, parameter tuning already done (checked in notebooks 02 & 03).

In [2]:
import json, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.stats.multitest import multipletests
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
%matplotlib inline

RANDOM_SEED    = 42
np.random.seed(RANDOM_SEED)

COST_BASE_PIPS  = 2.0   # spread only (pips)
COST_STRESS_PIPS= 4.0   # stressed: wider spread + slippage
RISK_PER_TRADE  = 1.0   # 1R

# Load all three datasets
base = pd.read_csv('../DATA/base_ml_4yr_processed.csv', parse_dates=['open_time'])
rb   = pd.read_csv('../DATA/rulebased_5yr_processed.csv', parse_dates=['open_time'])
scores = pd.read_csv('../DATA/ml_scores_all.csv', parse_dates=['open_time'])

base = base.sort_values('open_time').reset_index(drop=True)
rb   = rb.sort_values('open_time').reset_index(drop=True)

# Merge ML scores onto base
base = base.merge(scores[['pos_id','open_time','ml_score']], on=['pos_id','open_time'], how='left')

# Common evaluation window (holdout only)
HOLDOUT_START = '2025-12-01'
holdout_base = base[base['split'] == 'holdout'].copy()
holdout_rb   = rb[rb['split'] == 'holdout'].copy()

# ML holdout = top 50% scored trades from base holdout
ml_cutoff   = holdout_base['ml_score'].quantile(0.5)
holdout_ml  = holdout_base[holdout_base['ml_score'] >= ml_cutoff].copy()

print(f'Holdout Base      : {len(holdout_base):,} trades')
print(f'Holdout Rule-Based: {len(holdout_rb):,} trades')
print(f'Holdout ML (top50): {len(holdout_ml):,} trades')

Holdout Base      : 549 trades
Holdout Rule-Based: 191 trades
Holdout ML (top50): 275 trades


## 1. Apply Cost Adjustment

In [4]:
def apply_cost(df, cost_pips):
    df = df.copy()
    # Cost in R = extra pips / sl_pips
    df['cost_R']     = cost_pips / df['sl_pips'].replace(0, np.nan)
    df['result_R_net'] = df['result_R'] - df['cost_R'].fillna(0)
    return df

models = {}
for name, dset in [('Base', holdout_base), ('Rule-Based', holdout_rb), ('ML (top50%)', holdout_ml)]:
    models[name] = {
        'base' : apply_cost(dset, COST_BASE_PIPS),
        'stress': apply_cost(dset, COST_STRESS_PIPS),
    }

## 2. Core Metrics Table

In [6]:
def full_metrics(df, r_col='result_R_net', label=''):
    if len(df) == 0:
        return {}
    r       = df[r_col].dropna()
    n       = len(r)
    wr      = (r > 0).mean()
    exp_r   = r.mean()
    winners = r[r > 0].sum()
    losers  = r[r < 0].sum()
    pf      = winners / abs(losers) if losers != 0 else np.nan
    eq      = r.cumsum().reset_index(drop=True)
    dd      = eq - eq.cummax()
    mdd     = dd.min()

    # Calmar = annualised return / |MDD|
    days = (df['open_time'].max() - df['open_time'].min()).days
    ann_r = exp_r * n * 252 / max(days, 1)
    calmar = ann_r / abs(mdd) if mdd != 0 else np.nan

    # Worst month
    monthly = df.set_index('open_time')[r_col].resample('ME').sum()
    worst_m = monthly.min()
    avg_tpm = monthly.count() and n / (days / 30)

    # Longest losing streak
    max_ls = cur_ls = 0
    for rv in r:
        if rv < 0: cur_ls += 1; max_ls = max(max_ls, cur_ls)
        else: cur_ls = 0

    return dict(
        label=label, n=n, wr=round(wr,3), exp_r=round(exp_r,4),
        pf=round(pf,3), calmar=round(calmar,3) if calmar else np.nan,
        mdd=round(mdd,3), worst_month=round(worst_m,3),
        max_losing_streak=max_ls, avg_trades_month=round(avg_tpm,1)
    )

# Base cost
metrics_base  = [full_metrics(models[m]['base'], label=m) for m in models]
metrics_df    = pd.DataFrame(metrics_base).set_index('label')
print('=== Holdout Metrics — Base Cost ===')
display(metrics_df)

# Stressed cost
metrics_stress = [full_metrics(models[m]['stress'], label=m+' (stressed)') for m in models]
metrics_stress_df = pd.DataFrame(metrics_stress).set_index('label')
print('\n=== Holdout Metrics — Stressed Cost ===')
display(metrics_stress_df)

=== Holdout Metrics — Base Cost ===
               n     wr   exp_r     pf  calmar     mdd  worst_month  max_losing_streak  avg_trades_month
label                                                                                                   
Base         549  0.348  0.0320  1.048   0.347 -43.212      -21.631                 17              55.8
Rule-Based   191  0.366  0.1675  1.262   1.429 -19.131       -5.412                 10              19.4
ML (top50%)  275  0.371  0.0995  1.155   0.989 -23.627      -14.537                 18              28.0

=== Holdout Metrics — Stressed Cost ===
                          n     wr   exp_r     pf  calmar     mdd  worst_month  max_losing_streak  avg_trades_month
label                                                                                                              
Base (stressed)         549  0.348  0.0304  1.045   0.328 -43.451      -21.807                 17              55.8
Rule-Based (stressed)   191  0.366  0.1661  1.259 

## 3. Equity Curve Comparison

In [8]:
fig, axes = plt.subplots(2, 1, figsize=(14, 9))

colors = {'Base': 'steelblue', 'Rule-Based': 'darkorange', 'ML (top50%)': 'seagreen'}

for cost_key, ax, ttl in [('base', axes[0], 'Base Cost'), ('stress', axes[1], 'Stressed Cost')]:
    for name, dsets in models.items():
        dset = dsets[cost_key]
        if len(dset) == 0:
            continue
        eq = dset['result_R_net'].cumsum().reset_index(drop=True)
        ax.plot(dset['open_time'].values, eq.values,
                lw=1.8, label=name, color=colors[name])
    ax.axhline(0, color='k', lw=0.8)
    ax.set_title(f'Holdout Equity Curves — {ttl}')
    ax.set_ylabel('Cumulative R (net)')
    ax.legend()

plt.tight_layout()
plt.savefig('../DATA/04_equity_comparison.png', dpi=120)
plt.show()

## 4. Paired Bootstrap Significance Test

In [10]:
np.random.seed(RANDOM_SEED)
N_BOOT = 10_000

# Match on common pos_id for paired test
base_r = models['Base']['base'].set_index('pos_id')['result_R_net']
rb_r   = models['Rule-Based']['base'].set_index('pos_id')['result_R_net']
ml_r   = models['ML (top50%)']['base'].set_index('pos_id')['result_R_net']

paired_tests = []
for name_a, r_a, name_b, r_b in [
    ('Rule-Based', rb_r, 'Base',       base_r),
    ('ML (top50%)', ml_r, 'Base',      base_r),
    ('Rule-Based', rb_r, 'ML (top50%)', ml_r),
]:
    # Use all trades in each group (independent bootstrap since sets differ)
    obs_diff = r_a.mean() - r_b.mean()
    combined = np.concatenate([r_a.values, r_b.values])
    boot_diffs = []
    for _ in range(N_BOOT):
        s_a = np.random.choice(r_a.values, len(r_a), replace=True).mean()
        s_b = np.random.choice(r_b.values, len(r_b), replace=True).mean()
        boot_diffs.append(s_a - s_b)
    boot_diffs = np.array(boot_diffs)
    p_val = (boot_diffs <= 0).mean() if obs_diff > 0 else (boot_diffs >= 0).mean()
    ci_lo, ci_hi = np.percentile(boot_diffs, [2.5, 97.5])
    paired_tests.append(dict(
        comparison=f'{name_a} vs {name_b}',
        obs_diff=round(obs_diff, 5),
        ci_lo=round(ci_lo, 5), ci_hi=round(ci_hi, 5),
        raw_p=round(p_val, 4)
    ))

pair_df = pd.DataFrame(paired_tests)

# Benjamini-Hochberg correction for multiple comparisons
reject, pvals_corr, _, _ = multipletests(pair_df['raw_p'], method='fdr_bh')
pair_df['p_bh_corrected'] = pvals_corr.round(4)
pair_df['reject_H0']      = reject

print(f'Bootstrap significance (N={N_BOOT:,}), BH-corrected:')
display(pair_df)

Bootstrap significance (N=10,000), BH-corrected:
                  comparison  obs_diff    ci_lo    ci_hi   raw_p  p_bh_corrected  reject_H0
0         Rule-Based vs Base   0.13558 -0.11213  0.39010  0.1455           0.322      False
1        ML (top50%) vs Base   0.06749 -0.14372  0.27781  0.2635           0.322      False
2  Rule-Based vs ML (top50%)   0.06810 -0.21219  0.35770  0.3220           0.322      False


## 5. Bootstrap CI on Individual Model Expectancy

In [12]:
np.random.seed(RANDOM_SEED)
boot_results = []

for name, dsets in models.items():
    r = dsets['base']['result_R_net'].dropna().values
    if len(r) < 10:
        continue
    boots = [np.random.choice(r, len(r), replace=True).mean() for _ in range(N_BOOT)]
    ci_lo, ci_hi = np.percentile(boots, [2.5, 97.5])
    boot_results.append(dict(model=name, exp_r=r.mean(),
                             ci_lo=ci_lo, ci_hi=ci_hi,
                             above_zero_pct=(np.array(boots) > 0).mean()))

boot_df = pd.DataFrame(boot_results)
display(boot_df.round(5))

fig, ax = plt.subplots(figsize=(9, 4))
for i, row in boot_df.iterrows():
    color = 'steelblue' if row['ci_lo'] > 0 else ('darkorange' if row['ci_hi'] > 0 else 'crimson')
    ax.errorbar(row['exp_r'], row['model'],
                xerr=[[row['exp_r']-row['ci_lo']], [row['ci_hi']-row['exp_r']]],
                fmt='o', color=color, capsize=5, ms=8)
ax.axvline(0, color='k', lw=0.8, ls='--')
ax.set_title('Bootstrap 95% CI — Expectancy per Model (Holdout)')
ax.set_xlabel('Expectancy (R, net of costs)')
plt.tight_layout()
plt.savefig('../DATA/04_bootstrap_ci.png', dpi=120)
plt.show()

         model    exp_r    ci_lo    ci_hi  above_zero_pct
0         Base  0.03197 -0.08842  0.15473          0.7037
1   Rule-Based  0.16755 -0.04566  0.38803          0.9296
2  ML (top50%)  0.09945 -0.06636  0.27449          0.8730


## 6. Cost Sensitivity Analysis

In [14]:
cost_range = np.arange(0, 8.5, 0.5)
sensitivity = {name: [] for name in models}

for cost in cost_range:
    for name, dsets in models.items():
        dset = dsets['base'].copy()
        dset['result_R_net'] = dset['result_R'] - cost / dset['sl_pips'].replace(0, np.nan).fillna(1)
        sensitivity[name].append(dset['result_R_net'].mean())

fig, ax = plt.subplots(figsize=(10, 5))
for name, vals in sensitivity.items():
    ax.plot(cost_range, vals, lw=2, label=name, color=colors[name])
ax.axhline(0, color='k', lw=0.8, ls='--')
ax.set_title('Cost Sensitivity — Expectancy vs. Spread+Slippage (pips)')
ax.set_xlabel('Total Cost (pips)')
ax.set_ylabel('Expectancy (R)')
ax.legend()
plt.tight_layout()
plt.savefig('../DATA/04_cost_sensitivity.png', dpi=120)
plt.show()

## 7. Monthly Heatmap Comparison

In [16]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (name, dsets) in zip(axes, models.items()):
    dset = dsets['base']
    if len(dset) == 0:
        ax.set_visible(False); continue
    monthly = dset.set_index('open_time')['result_R_net'].resample('ME').agg(['sum','count'])
    monthly.index = monthly.index.to_period('M')
    monthly['year']  = monthly.index.year
    monthly['month'] = monthly.index.month
    pivot = monthly.pivot(index='year', columns='month', values='sum')
    sns.heatmap(pivot, annot=True, fmt='.1f', cmap='RdYlGn', center=0,
                linewidths=0.3, ax=ax, cbar=False)
    ax.set_title(f'{name} — Monthly R')
    ax.set_xlabel('Month'); ax.set_ylabel('Year')

plt.tight_layout()
plt.savefig('../DATA/04_monthly_heatmap.png', dpi=120)
plt.show()

## 8. Decision Rule Application

In [18]:
print('='*60)
print('DECISION RULE — Applied to Holdout (Base Cost)')
print('='*60)
print('Criteria:')
print('  1. Bootstrap 95% CI lower bound > 0')
print('  2. Profit Factor > 1.1')
print('  3. Max Drawdown > -15 R')
print('  4. Avg trades/month >= 20')
print('  Tiebreak: highest Calmar ratio')
print()

decision = []
for _, row in boot_df.iterrows():
    m_name = row['model']
    m_row  = metrics_df.loc[m_name] if m_name in metrics_df.index else None
    if m_row is None:
        continue
    criteria = {
        'CI_lo>0'       : row['ci_lo'] > 0,
        'PF>1.1'        : m_row['pf'] > 1.1,
        'MDD>-15R'      : m_row['mdd'] > -15,
        'Trades>=20/mo' : m_row['avg_trades_month'] >= 20,
    }
    passes_all = all(criteria.values())
    print(f'  [{m_name}]')
    for k, v in criteria.items():
        print(f'    {"✓" if v else "✗"} {k}')
    print(f'    → PASSES all criteria: {passes_all}')
    print(f'    → Calmar: {m_row["calmar"]}')
    print()
    decision.append(dict(model=m_name, passes=passes_all,
                         calmar=m_row['calmar'], **criteria))

decision_df = pd.DataFrame(decision)
winners     = decision_df[decision_df['passes']].sort_values('calmar', ascending=False)
if len(winners):
    print(f'RECOMMENDATION: {winners.iloc[0]["model"]} (highest Calmar among passing models)')
else:
    print('NO MODEL passes all criteria on holdout. Review thresholds or extend data.')

DECISION RULE — Applied to Holdout (Base Cost)
Criteria:
  1. Bootstrap 95% CI lower bound > 0
  2. Profit Factor > 1.1
  3. Max Drawdown > -15 R
  4. Avg trades/month >= 20
  Tiebreak: highest Calmar ratio

  [Base]
    ✗ CI_lo>0
    ✗ PF>1.1
    ✗ MDD>-15R
    ✓ Trades>=20/mo
    → PASSES all criteria: False
    → Calmar: 0.347

  [Rule-Based]
    ✗ CI_lo>0
    ✓ PF>1.1
    ✗ MDD>-15R
    ✗ Trades>=20/mo
    → PASSES all criteria: False
    → Calmar: 1.429

  [ML (top50%)]
    ✗ CI_lo>0
    ✓ PF>1.1
    ✗ MDD>-15R
    ✓ Trades>=20/mo
    → PASSES all criteria: False
    → Calmar: 0.989

NO MODEL passes all criteria on holdout. Review thresholds or extend data.


## 9. Full Summary Dashboard

In [20]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
metric_map = [
    ('wr',          'Win Rate',         [0, 1]),
    ('exp_r',       'Expectancy (R)',    None),
    ('pf',          'Profit Factor',     None),
    ('mdd',         'Max Drawdown (R)',  None),
    ('calmar',      'Calmar Ratio',      None),
    ('avg_trades_month', 'Avg Trades/Month', None),
]

model_names = list(models.keys())
cmap_colors = ['steelblue','darkorange','seagreen']

for ax, (metric, title, ylim) in zip(axes.flat, metric_map):
    vals = [metrics_df.loc[m, metric] if m in metrics_df.index else np.nan for m in model_names]
    ax.bar(model_names, vals, color=cmap_colors, alpha=0.85)
    ax.set_title(title)
    ax.set_ylabel(metric)
    ax.tick_params(axis='x', rotation=15)
    if ylim:
        ax.set_ylim(ylim)
    if metric in ['exp_r','mdd','calmar']:
        ax.axhline(0, color='k', lw=0.8, ls='--')
    if metric == 'pf':
        ax.axhline(1, color='k', lw=0.8, ls='--')

plt.suptitle('Model Comparison Summary — Holdout Period', fontsize=15, y=1.01)
plt.tight_layout()
plt.savefig('../DATA/04_summary_dashboard.png', dpi=120)
plt.show()